<a href="https://colab.research.google.com/github/dice-group/HTYLLM2-PG/blob/ice-breaker/LLMGerman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install & Imports

In [ ]:
# Install  libraries
!pip install transformers datasets tokenizers accelerate -q

import math
import os
import torch

from datasets import load_dataset, Dataset
from tokenizers import ByteLevelBPETokenizer
from transformers import (
    GPT2Config,                      # model architecture config
    GPT2LMHeadModel,                 # GPT-2 model class
    PreTrainedTokenizerFast,         # tokenizer wrapper for Trainer
    Trainer,                         # training loop
    TrainingArguments,               # training hyperparameters
    DataCollatorForLanguageModeling, # batches data for CLM
)

#Configuration

In [ ]:
# we can use also "fr"
LANGUAGE = "de" #"fr"
WIKI_SNAPSHOT = "20231101.de" #"20230920.fr"
# on vm encrease all the parameter
# Data size
NUM_ARTICLES = 50_000    # Wikipedia articles to use
VOCAB_SIZE = 20_000      # tokenizer vocabulary size
MAX_LENGTH = 256         # tokens per training sample

#this has not to be changed
# Model architecture GPT-2
N_EMBD = 384             # the embedding dimension can be tested
N_LAYER = 6              # transformer layers could also be tested
N_HEAD = 6               # attention heads per layer

# Training settings
NUM_EPOCHS = 2           # passes through the dataset
BATCH_SIZE = 16          #samples per GPU step
LEARNING_RATE = 3e-4     #Regulatization
WARMUP_STEPS = 200       # gradual LR ramp-up at start

GRAD_ACCUM_STEPS = 1
DATALOADER_WORKERS = 2


# Output folders
TOKENIZER_DIR = f"tokenizer_{LANGUAGE}"
MODEL_DIR = f"model_{LANGUAGE}"

#Load Data

In [ ]:
# Stream Wikipedia so we don't download the full dataset
raw_dataset = load_dataset(
    "wikimedia/wikipedia",
    WIKI_SNAPSHOT,
    split="train",
    streaming=True, # mukund on vm has to be changed into streaming=False
)

# Pulling the first N articles into memory
texts = [example["text"] for example in raw_dataset.take(NUM_ARTICLES)]

total_mb = sum(len(t) for t in texts) / 1_000_000
print(f"Articles loaded: {len(texts):,}")
print(f"Corpus size: {total_mb:.1f} MB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Articles loaded: 50,000
Corpus size: 657.1 MB


#Train Tokenizer

In [ ]:
os.makedirs(TOKENIZER_DIR, exist_ok=True)

#  Byte-Level BPE tokenizer
bpe = ByteLevelBPETokenizer()
bpe.train_from_iterator(
    texts,
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=["<s>", "</s>", "<unk>", "<pad>"],  # required control tokens
)
bpe.save(f"{TOKENIZER_DIR}/tokenizer.json")

# Wrapping in HuggingFace interface (From This will trainer work)
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=f"{TOKENIZER_DIR}/tokenizer.json",
    bos_token="<s>",
    eos_token="</s>",
    unk_token="<unk>",
    pad_token="<pad>",
)

print(f"Vocab size: {tokenizer.vocab_size:,}")

Vocab size: 20,000


#Prepare Dataset

In [ ]:
# Converting each samples to token IDs
def tokenize(batch):
    return tokenizer(batch["text"], max_length=MAX_LENGTH, truncation=True)

hf_dataset = Dataset.from_dict({"text": texts})
tokenized_dataset = hf_dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"],   # droping raw text, will keep only token IDs (for embadding)
)

print(f"Tokenized samples: {len(tokenized_dataset):,}")

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Tokenized samples: 50,000


#Define Model

In [ ]:
# GPT-2 architecture with random weights
config = GPT2Config(
    vocab_size=VOCAB_SIZE,
    n_positions=MAX_LENGTH,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
)

model = GPT2LMHeadModel(config)
print(f"Model parameters: {model.num_parameters():,}")

Model parameters: 18,425,856


#Train

In [ ]:
# Collator builds training batches; mlm=False means Causal LM when is this true than we have Masked LM
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,
    fp16=True,                  # half precision -> 2x faster on GPU on vm has be changed into False
    report_to="none",
    # ddp_find_unused_parameters=False #only on vm
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

train_result = trainer.train()

# Perplexity
perplexity = math.exp(train_result.training_loss)
print(f"Final training loss: {train_result.training_loss:.4f}")
print(f"Perplexity: {perplexity:.2f}")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,8.913569
200,7.570575
300,7.101555
400,6.762739
500,6.534329
600,6.377456
700,6.225361
800,6.104555
900,5.997556
1000,5.924100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final training loss: 5.2121
Perplexity: 183.48


#Generate Text

In [ ]:
model.eval()  # disable dropout for inference
#  prompt here:

prompt = " ChatGPT "
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

# Sampling-based generation with top-k and top-p filtering
output = model.generate(
    input_ids,
    max_length=50,
    do_sample=True,
    top_k=50,                              # keep only top 50 likely tokens
    top_p=0.95,                            # nucleus sampling threshold
    temperature=0.8,                       # randomness (lower = more focused)
    pad_token_id=tokenizer.pad_token_id,
)

print(f"Prompt:    {prompt}")
print(f"Generated: {tokenizer.decode(output[0], skip_special_tokens=True)}")

Prompt:     Ibrahim ist person 
Generated:  Ibrahim ist person  in der griechischen Mythologie eine der griechischen Mythologie. Die Art ist eine der Hauptgott und der griechischen Mythologie.

Handlung 

Der Vater des Gulf von Dlah, der eine von drei Brüder 
